In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import date, datetime, timedelta
from pyspark.sql.window import Window

GOLD_BD = "gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{GOLD_BD}")

df_filmes = spark.table("workspace.silver.tb_info_filmes")
janela = Window.orderBy("id_filme")

#Cria gold.dim_movies
df_dim_movies = df_filmes.select(
    F.row_number().over(janela).try_cast("bigint").alias("sk_movie_id"),
    F.col("id_filme").try_cast("string"),
    F.col("titulo").try_cast("string"),
    F.col("data_lancamento").try_cast("date"),
    F.col("ano_lancamento").try_cast("int"),
    F.col("duracao_minutos").try_cast("int"),
    F.col("idioma_original").try_cast("string"),
    F.col("status_filme").try_cast("string"),
    F.col("sinopse").try_cast("string"),
)

display(df_dim_movies)

df_dim_movies.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"workspace.{GOLD_BD}.dim_movies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
6,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null
7,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
8,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
9,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
10,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."


In [0]:


df_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_metricas   = spark.table("workspace.silver.tb_metricas_engajamento")
df_dim_movies = spark.table(f"workspace.{GOLD_BD}.dim_movies")

df_fact_movie_performance = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_metricas,   on="id_filme", how="left")
    .select(
        F.col("sk_movie_id"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

display(df_fact_movie_performance)

df_fact_movie_performance.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"workspace.{GOLD_BD}.fact_movies_performance")

sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,0.00,null,null,0.00,1.132,0.0,0,6.8,27
2,null,null,0.00,null,null,0.00,0.6,0.0,0,null,40
3,null,null,0.00,null,null,0.00,0.6,0.0,0,4.7,10
4,null,null,0.00,null,null,0.00,1.169,0.0,0,4.8,16
5,null,null,0.00,null,null,0.00,0.6,0.0,0,7.2,15
6,null,null,0.00,null,null,0.00,0.615,0.0,0,null,25
7,null,null,0.00,null,null,0.00,0.6,0.0,0,5.9,10
8,4700000.00,null,-4700000.00,23885400.00,null,-23885400.00,1.489,6.75,6,6.2,382
9,null,null,0.00,null,null,0.00,0.6,0.0,0,7.7,25
10,6000000.00,null,-6000000.00,30492000.00,null,-30492000.00,13.212,6.8,15,null,236


In [0]:
df_generos = spark.table("workspace.silver.tb_generos")

df_dim_genres = (
    df_generos
    .select(F.col("genres").alias("nome_genero"))
    .filter(F.col("nome_genero").isNotNull())
    .distinct()
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

display(df_dim_genres)

df_dim_genres.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_genres")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_genre_id,nome_genero
1,"""""\""""Father of Indian football""""\""""\""""""""""Romance History""SVF Entertainment"
2,"""99 were safe.\""""\""""""""""Horror"
3,"""Bahar Ebrahim"
4,"""Citra Sinema"
5,"""Dad\"""".\"""""""""""
6,"""Doo Hyeon seeks revenge and announces the return of the real \""""Goblin\""""\"""".\"""""""""""
7,"""Drama"
8,"""English"
9,"""German"
10,"""I can always rely on my friends.\""""\""""""""""Animation"


In [0]:
dim = spark.table("workspace.gold.dim_genres")

print("Gêneros:", dim.count(), "| Distintos:", dim.select("nome_genero").distinct().count())

display(
    dim.groupBy(F.lower(F.trim("nome_genero")).alias("genero_normalizado"))
       .count().filter("count > 1")
)

Gêneros: 2694 | Distintos: 2694


genero_normalizado,count
